# Train CIFAR-100 → DiNN-style 3072 → 100 → 100

Trains a small bipolar MLP on CIFAR-100 **fine** labels and exports its weights as plain CSV files compatible with this repo's C++ FHE inference pipeline (see `cifar10/main.cpp` and `src/io/csv.cpp`).

Architecture: `3072 → 100 → 100`

* **Input:** 32 × 32 × 3 RGB image, flattened in HWC order, with each byte thresholded at 127 to {-1, +1}.
* **Hidden:** `Linear(3072 → 100)` followed by a hard-sign activation (straight-through estimator during training).
* **Output:** `Linear(100 → 100)` — 100 fine-grained CIFAR-100 classes.

## 1. Setup and imports

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Configuration

Everything tunable lives in `CONFIG`. The default `output_dir` is `cifar100/` (relative to the notebook's working directory) so the artifacts land in a sibling of the existing `cifar10/`-style folders when this notebook is run from the repo root or via `make up`.

In [ ]:
CONFIG = {
    "epochs": 30,
    "batch_size": 256,
    "lr": 1e-3,
    "weight_decay": 0.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 0,
    "data_dir": "./data",
    "output_dir": "cifar100",
    "in_dim": 32 * 32 * 3,
    "hidden_dim": 100,
    "num_classes": 100,
}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG["seed"])
DEVICE = torch.device(CONFIG["device"])
print("Device:", DEVICE)
print("Output dir:", Path(CONFIG["output_dir"]).resolve())

## 3. CIFAR-100 dataset loading

We download CIFAR-100 with no transform so we keep the raw PIL images and can preprocess them ourselves — the same way the C++ side does, byte for byte.

`torchvision.datasets.CIFAR100` defaults to **fine** labels (0..99). Coarse (super-class) labels are not used here.

In [ ]:
train_pil = datasets.CIFAR100(
    root=CONFIG["data_dir"], train=True, download=True, transform=None
)
test_pil = datasets.CIFAR100(
    root=CONFIG["data_dir"], train=False, download=True, transform=None
)

print("Train:", len(train_pil), " Test:", len(test_pil))
print("Number of fine classes:", len(train_pil.classes))

_img, _label = train_pil[0]
print("Sample:", _img.size, _img.mode, "label =", _label, "(", train_pil.classes[_label], ")")

## 4. C++-compatible preprocessing

The C++ inference path does:

```cpp
// stb_image returns flat HWC interleaved RGB bytes when called with channels=3
unsigned char* data = stbi_load(path, &w, &h, &ch, /*channels=*/3);
for (int i = 0; i < 32*32*3; ++i) out[i] = (data[i] > 127) ? +1 : -1;
```

We replicate that exactly:

1. Convert PIL → NumPy `uint8` array of shape `(H, W, 3)` in **HWC** order.
2. Threshold each raw pixel byte at 127.
3. Map `> 127 → +1.0`, otherwise `-1.0`.
4. Flatten in HWC order (row, then column, then channel) to length 3072.

We deliberately **do not** use PyTorch's default `ToTensor` / CHW conversion — that would put all R bytes first, then all G, then all B, which is **not** the layout the C++ inference loop expects.

In [ ]:
def pil_to_bipolar_hwc(img) -> np.ndarray:
    """PIL → NumPy HWC uint8 → threshold at 127 → {-1, +1} float32, flat length 3072."""
    arr = np.asarray(img.convert("RGB"), dtype=np.uint8)  # (H, W, 3) HWC
    assert arr.shape == (32, 32, 3), f"unexpected shape {arr.shape}"
    bipolar = np.where(arr > 127, np.float32(1.0), np.float32(-1.0))
    flat = bipolar.reshape(-1)  # HWC row-major: (h, w, c) → single 3072-vector
    assert flat.shape == (3072,)
    return flat


def encode_dataset(pil_dataset) -> tuple[np.ndarray, np.ndarray]:
    n = len(pil_dataset)
    X = np.empty((n, 3072), dtype=np.float32)
    y = np.empty((n,), dtype=np.int64)
    for i in range(n):
        img, label = pil_dataset[i]
        X[i] = pil_to_bipolar_hwc(img)
        y[i] = label
    return X, y


X_train, y_train = encode_dataset(train_pil)
X_test, y_test = encode_dataset(test_pil)

print("X_train:", X_train.shape, X_train.dtype, "min/max =", X_train.min(), X_train.max())
print("X_test :", X_test.shape,  X_test.dtype,  "min/max =", X_test.min(),  X_test.max())
print("y_train classes:", len(np.unique(y_train)), "  y_test classes:", len(np.unique(y_test)))

# Sanity: every value is exactly +1 or -1.
assert set(np.unique(X_train)).issubset({-1.0, 1.0})
assert set(np.unique(X_test)).issubset({-1.0, 1.0})

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
print("train batches:", len(train_loader), " test batches:", len(test_loader))

## 5. Model definition

DiNN-style MLP:

* `fc1 = Linear(3072, 100)`
* hard-sign activation: `+1` if `x ≥ 0`, else `-1`
* `fc2 = Linear(100, 100)`

The hard-sign has zero gradient almost everywhere, so during training we wrap it in a **straight-through estimator**: the forward pass is the true sign, but the backward pass passes the gradient through, clamped to the region `|x| ≤ 1` (the standard BinaryNet / DiNN trick). This stops weights from blowing up while still letting the upstream `fc2` shape `fc1`'s pre-activations.

In [ ]:
class SignSTE(torch.autograd.Function):
    """Forward = hard sign with +1 at zero. Backward = clamped identity (STE)."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.where(x >= 0, torch.ones_like(x), -torch.ones_like(x))

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        # Pass gradient through, but kill it where |x| > 1 to keep fc1 weights tame.
        grad_input = grad_output * (x.abs() <= 1.0).to(grad_output.dtype)
        return grad_input


def sign_ste(x: torch.Tensor) -> torch.Tensor:
    return SignSTE.apply(x)


class DiNN(nn.Module):
    """3072 → hidden → num_classes with a bipolar hidden activation."""

    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc1(x)
        h = sign_ste(h)
        return self.fc2(h)


model = DiNN(CONFIG["in_dim"], CONFIG["hidden_dim"], CONFIG["num_classes"]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("Trainable params:", n_params)

## 6. Training loop

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
)
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * xb.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += xb.size(0)
    return loss_sum / total, correct / total


# Per-epoch history so we can plot loss / accuracy curves later.
history = {
    "epoch":      [],
    "train_loss": [],
    "train_acc":  [],
    "test_loss":  [],
    "test_acc":   [],
}

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_n = 0
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
        epoch_correct += (logits.argmax(dim=1) == yb).sum().item()
        epoch_n += xb.size(0)

    train_loss = epoch_loss / epoch_n
    train_acc  = epoch_correct / epoch_n
    test_loss, test_acc = evaluate(model, test_loader, DEVICE)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"epoch {epoch:3d}/{CONFIG['epochs']}  "
        f"train_loss={train_loss:.4f}  train_acc={train_acc*100:.2f}%  "
        f"test_loss={test_loss:.4f}  test_acc={test_acc*100:.2f}%"
    )

## 6b. Training curves

Two side-by-side plots from the per-epoch `history` collected above:

* **Loss vs epoch** — train loss (running average over the epoch) and test loss.
* **Accuracy vs epoch** — train accuracy (computed on the live, mid-training predictions) and test accuracy.

The train-loss curve is a *running* average across the epoch's mini-batches, so it lags slightly behind the true end-of-epoch value — that's expected and is the standard way to plot it cheaply.

In [ ]:
import matplotlib.pyplot as plt

epochs = history["epoch"]

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_loss.plot(epochs, history["train_loss"], label="train loss", marker="o", markersize=3)
ax_loss.plot(epochs, history["test_loss"],  label="test loss",  marker="o", markersize=3)
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title("CIFAR-100 DiNN: loss vs epoch")
ax_loss.grid(True, alpha=0.3)
ax_loss.legend()

ax_acc.plot(epochs, [a * 100 for a in history["train_acc"]], label="train acc", marker="o", markersize=3)
ax_acc.plot(epochs, [a * 100 for a in history["test_acc"]],  label="test acc",  marker="o", markersize=3)
ax_acc.set_xlabel("epoch")
ax_acc.set_ylabel("accuracy (%)")
ax_acc.set_title("CIFAR-100 DiNN: accuracy vs epoch")
ax_acc.grid(True, alpha=0.3)
ax_acc.legend()

best_epoch = int(np.argmax(history["test_acc"])) + 1
best_acc   = max(history["test_acc"]) * 100
print(f"Best test accuracy: {best_acc:.2f}% at epoch {best_epoch} (chance = 1.00%).")

fig.tight_layout()
plt.show()

## 7. Evaluation

In [ ]:
final_train_loss, final_train_acc = evaluate(model, train_loader, DEVICE)
final_test_loss,  final_test_acc  = evaluate(model, test_loader,  DEVICE)
print(f"Final train accuracy: {final_train_acc*100:.2f}%  (loss {final_train_loss:.4f})")
print(f"Final test  accuracy: {final_test_acc*100:.2f}%  (loss {final_test_loss:.4f})")

## 8. CSV export

We export raw float weights (no quantization), no headers, in shapes:

| File | Shape |
|---|---|
| `cifar100_weights_W1.csv` | `(100, 3072)` |
| `cifar100_weights_b1.csv` | `(100,)` |
| `cifar100_weights_W2.csv` | `(100, 100)` |
| `cifar100_weights_b2.csv` | `(100,)` |

`torch.nn.Linear.weight` is already stored as `(out_features, in_features)`, which matches the C++ `LoadCsv2D(path, expected_in, expected_out)` preferred orientation, so we can dump the parameters directly without transposing.

In [ ]:
output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

W1 = model.fc1.weight.detach().cpu().numpy()  # (100, 3072)
b1 = model.fc1.bias.detach().cpu().numpy()    # (100,)
W2 = model.fc2.weight.detach().cpu().numpy()  # (100, 100)
b2 = model.fc2.bias.detach().cpu().numpy()    # (100,)

paths = {
    "W1": output_dir / "cifar100_weights_W1.csv",
    "b1": output_dir / "cifar100_weights_b1.csv",
    "W2": output_dir / "cifar100_weights_W2.csv",
    "b2": output_dir / "cifar100_weights_b2.csv",
}

# 2D matrices: comma-separated rows, no headers; 1D vectors: one value per line.
np.savetxt(paths["W1"], W1, delimiter=",", fmt="%.8f")
np.savetxt(paths["W2"], W2, delimiter=",", fmt="%.8f")
np.savetxt(paths["b1"], b1, fmt="%.8f")
np.savetxt(paths["b2"], b2, fmt="%.8f")

for name, p in paths.items():
    print(f"{name}: {p}  ({p.stat().st_size} bytes)")

## 9. Export verification

Reload the CSVs with NumPy and check shapes plus value parity (within the precision of the `%.8f` formatter we used).

In [ ]:
W1_rl = np.loadtxt(paths["W1"], delimiter=",")
b1_rl = np.loadtxt(paths["b1"])
W2_rl = np.loadtxt(paths["W2"], delimiter=",")
b2_rl = np.loadtxt(paths["b2"])

assert W1_rl.shape == (100, 3072), W1_rl.shape
assert b1_rl.shape == (100,),      b1_rl.shape
assert W2_rl.shape == (100, 100),  W2_rl.shape
assert b2_rl.shape == (100,),      b2_rl.shape

print("max |W1 - W1_rl|:", float(np.max(np.abs(W1.astype(np.float64) - W1_rl))))
print("max |b1 - b1_rl|:", float(np.max(np.abs(b1.astype(np.float64) - b1_rl))))
print("max |W2 - W2_rl|:", float(np.max(np.abs(W2.astype(np.float64) - W2_rl))))
print("max |b2 - b2_rl|:", float(np.max(np.abs(b2.astype(np.float64) - b2_rl))))
print("Shapes OK.")

## 10. Exported-model sanity check

Run inference using **the reloaded CSVs** with the exact bipolar pipeline that the C++ side will use:

```cpp
hidden[j] = (b1[j] + sum_i W1[j][i] * pixels[i] >= 0) ? +1 : -1;
scores[j] = b2[j] + sum_i W2[j][i] * hidden[i];
predicted = argmax(scores);
```

Then compare against the live PyTorch model. Predictions should agree on (approximately) every test sample — any disagreement is just `%.8f` rounding around hidden-layer pre-activations near 0.

In [ ]:
def numpy_inference(X: np.ndarray, W1, b1, W2, b2) -> np.ndarray:
    h_pre = X @ W1.T + b1                  # (N, 100)
    h     = np.where(h_pre >= 0, 1.0, -1.0)  # bipolar hard-sign, matches C++
    scores = h @ W2.T + b2                 # (N, 100)
    return np.argmax(scores, axis=1)


csv_pred = numpy_inference(X_test, W1_rl, b1_rl, W2_rl, b2_rl)
csv_acc  = float((csv_pred == y_test).mean())

model.eval()
torch_pred_chunks = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        torch_pred_chunks.append(logits.argmax(dim=1).cpu().numpy())
torch_pred = np.concatenate(torch_pred_chunks)
torch_acc  = float((torch_pred == y_test).mean())

agreement = float((csv_pred == torch_pred).mean())

print(f"PyTorch test accuracy:    {torch_acc*100:.2f}%")
print(f"Reloaded CSV accuracy:    {csv_acc*100:.2f}%")
print(f"Prediction agreement:     {agreement*100:.2f}%")

## 11. Summary

**Where the CSVs were written:** `CONFIG["output_dir"]` (default `cifar100/`), as four files with no headers:

| File | Shape | Source tensor | Meaning |
|---|---|---|---|
| `cifar100_weights_W1.csv` | `100 × 3072` | `model.fc1.weight` | First Linear layer weights, in `(out, in)` order. Row `j` is the weight vector for hidden neuron `j`. |
| `cifar100_weights_b1.csv` | `100`        | `model.fc1.bias`   | First Linear layer biases, one per hidden neuron. |
| `cifar100_weights_W2.csv` | `100 × 100`  | `model.fc2.weight` | Second Linear layer weights, in `(out, in)` order. Row `k` is the weight vector for output class `k`. |
| `cifar100_weights_b2.csv` | `100`        | `model.fc2.bias`   | Second Linear layer biases, one per output class. |

**How these map to a future `cifar100/main.cpp`:** mirror `cifar10/main.cpp` and just change three constants plus the weight paths:

```cpp
static constexpr int IN_DIM  = 3072;  // 32 * 32 * 3, same as CIFAR-10
static constexpr int HID_DIM = 100;   // was 30 in cifar10
static constexpr int OUT_DIM = 100;   // was 10  in cifar10  (CIFAR-100 has 100 classes)

auto W1 = io::LoadCsv2D("../cifar100_weights_W1.csv", IN_DIM,  HID_DIM);
auto b1 = io::LoadCsv1D("../cifar100_weights_b1.csv");
auto W2 = io::LoadCsv2D("../cifar100_weights_W2.csv", HID_DIM, OUT_DIM);
auto b2 = io::LoadCsv1D("../cifar100_weights_b2.csv");

auto pixels = io::LoadImageBipolar(argv[1], IN_DIM, /*channels=*/3);
```

Because we exported the weights as `(out, in)` (the C++ preferred orientation), `LoadCsv2D` will accept them without needing to transpose.

> **Naming reminder.** `CIFAR-100` here means **100 output classes** (with 100 hidden neurons in this particular topology). That's different from `MNIST_100`, where the `100` refers to the **hidden-layer width** and there are still only 10 output classes.